# RT-DETRv2 Fine-tuning on Custom Thermal Drone Dataset

Bu notebook, RT-DETRv2 modelini yerel (local) bir özel veri kümesi üzerinde ince ayar (fine-tune) eder.

- **Veri kümesi:** `dataset/dataset_augmented/` altındaki COCO formatındaki termal dron görüntüleri.
- **Sınıflar:** `Person`, `Car`, `OtherVehicle` (3 sınıf).
- **Kanal sayısı:** 3 (RGB olarak yüklenir — termal görüntüler 3 kanala genişletilir).
- **Augmentation:** Uygulanmaz — veri kümesi zaten çevrimdışı olarak augment edilmiştir.

## Ortamın kurulumu

Gerekli kütüphaneler: `transformers`, `timm`, `torchvision`, `Pillow`, `numpy`, `accelerate`, `albumentations` (sadece kimlik / bbox clipping için), `torchmetrics`. Ortamınızda kuruluysa aşağıdaki hücreyi atlayabilirsiniz.

In [ ]:
# Yerel ortamda zaten kuruluysa bu hücreyi atlayın.
# !pip install -q -U transformers timm torchvision Pillow "numpy==1.*"
# !pip install -q -U accelerate "albumentations==1.4.6" torchmetrics

Global sabitler: model checkpoint'i, giriş boyutu ve veri kümesi yolu.

In [ ]:
from pathlib import Path

# RT-DETRv2 R18 (daha küçük backbone — edge cihazlar için).
checkpoint = "PekingU/rtdetr_v2_r18vd"
image_size = 640

# Notebook'un bulunduğu klasöre göre veri kümesi yolu:
# models/rtdetr_v2_r18vd/<notebook>  ->  ../../dataset/dataset_augmented
DATASET_ROOT = Path("../../dataset/dataset_augmented").resolve()
TRAIN_DIR = DATASET_ROOT / "train"
VAL_DIR   = DATASET_ROOT / "val"
TEST_DIR  = DATASET_ROOT / "test"

assert TRAIN_DIR.exists(), f"Veri kümesi bulunamadı: {TRAIN_DIR}"
print("Dataset root:", DATASET_ROOT)

## Veri kümesini yükleme

Veri COCO formatında: her split için `images/` klasörü ve `_annotations.coco.json` dosyası. Kendi `CocoDetectionDataset` sınıfımızı kullanıyoruz (Hugging Face Datasets gerekmez).

In [ ]:
import json

def load_coco(split_dir):
    ann_file = split_dir / "_annotations.coco.json"
    with open(ann_file) as f:
        coco = json.load(f)
    return coco

train_coco = load_coco(TRAIN_DIR)
val_coco   = load_coco(VAL_DIR)
test_coco  = load_coco(TEST_DIR)

# Kategori eşlemeleri (her 3 split aynı kategorileri paylaşır)
categories = sorted(train_coco["categories"], key=lambda c: c["id"])
id2label = {c["id"]: c["name"] for c in categories}
label2id = {v: k for k, v in id2label.items()}

print("Classes:", id2label)
print(f"Train: {len(train_coco['images'])} images, {len(train_coco['annotations'])} annotations")
print(f"Val:   {len(val_coco['images'])} images,   {len(val_coco['annotations'])} annotations")
print(f"Test:  {len(test_coco['images'])} images,  {len(test_coco['annotations'])} annotations")

Bir örneği görselleştirerek verinin ve bounding box'ların doğru yüklendiğini kontrol edelim.

In [ ]:
sample_img = train_coco["images"][0]
sample_anns = [a for a in train_coco["annotations"] if a["image_id"] == sample_img["id"]]
print("Sample image:", sample_img)
print(f"Number of annotations for this image: {len(sample_anns)}")
print("First annotation:", sample_anns[0] if sample_anns else None)

Her örnek aşağıdaki alanlara sahiptir:
- `image_id`: görüntü kimliği
- `file_name`: `images/` altındaki dosya adı
- `width`, `height`: görüntü boyutu
- `annotations`: bbox'lar (COCO formatı: `[x, y, width, height]`) ve `category_id`

RT-DETRv2 COCO bbox formatını beklediği için dönüşüm gerekmez. Aşağıda örneği çizip görselleştirelim.

In [ ]:
import numpy as np
from PIL import Image, ImageDraw

image = Image.open(TRAIN_DIR / "images" / sample_img["file_name"]).convert("RGB")
draw = ImageDraw.Draw(image)
for ann in sample_anns:
    x, y, w, h = ann["bbox"]
    class_idx = ann["category_id"]
    draw.rectangle((x, y, x + w, y + h), outline="red", width=2)
    draw.text((x, y), id2label[class_idx], fill="white")

image

Bounding box'lar COCO formatında (`x_min, y_min, width, height`) yüklenir ve RT-DETRv2'ye bu formatta verilir.

## Ön işleme

`AutoImageProcessor`, modelin beklediği normalizasyon, yeniden boyutlandırma ve hedef formatını uygular. Eğitim sırasında kullandığımız checkpoint ile aynı image processor'ı yüklüyoruz.

In [ ]:
from transformers import AutoImageProcessor

image_processor = AutoImageProcessor.from_pretrained(
    checkpoint,
    do_resize=True,
    size={"width": image_size, "height": image_size},
    use_fast=True,
)

Veri kümesi **zaten augment edilmiş** olduğu için ek bir augmentation uygulanmaz. Yine de Albumentations üzerinden bir kimlik (no-op) dönüşümü kullanıyoruz — bu, bbox'ların görüntü sınırlarına kırpılması ve geçersiz kutuların (alan < 1 piksel) filtrelenmesi için faydalıdır.

In [ ]:
import albumentations as A

# Augmentation uygulanmaz; sadece bbox clip + geçersiz bbox filtresi.
train_transform = A.Compose(
    [A.NoOp()],
    bbox_params=A.BboxParams(
        format="coco",
        label_fields=["category"],
        clip=True,
        min_area=1,
        min_width=1,
        min_height=1,
    ),
)
validation_transform = train_transform

Veri kümesinin ön işleme çıktısını doğrulamak için COCO anotasyonlarını modelin beklediği formata çevirecek bir `Dataset` sınıfı tanımlıyoruz.

In [ ]:
# Dataset zaten augment edildiği için ek augmentation görselleştirmesine gerek yok.

`image_processor` anotasyonları şu formatta bekler: `{'image_id': int, 'annotations': List[Dict]}`. Aşağıdaki `CocoDetectionDataset` sınıfı diskten görüntü ve COCO anotasyonlarını okur, bunları processor için formatlar.

In [ ]:
from collections import defaultdict
from torch.utils.data import Dataset


class CocoDetectionDataset(Dataset):
    """COCO formatındaki yerel bir veri kümesinden görüntü ve anotasyon yükleyen sınıf."""

    def __init__(self, split_dir, coco, image_processor, transform=None):
        self.images_dir = Path(split_dir) / "images"
        self.image_processor = image_processor
        self.transform = transform

        # Geçerli görüntü dosyası olanları tut (bozuk / eksik dosyaları atla)
        self.images = {img["id"]: img for img in coco["images"]}

        annotations_by_image = defaultdict(list)
        for ann in coco["annotations"]:
            annotations_by_image[ann["image_id"]].append(ann)
        self.annotations_by_image = annotations_by_image

        self.image_ids = [
            img_id for img_id in self.images
            if (self.images_dir / self.images[img_id]["file_name"]).exists()
        ]

    @staticmethod
    def format_image_annotations_as_coco(image_id, categories, boxes):
        annotations = []
        for category, bbox in zip(categories, boxes):
            annotations.append({
                "image_id": image_id,
                "category_id": int(category),
                "bbox": list(bbox),
                "iscrowd": 0,
                "area": float(bbox[2]) * float(bbox[3]),
            })
        return {"image_id": image_id, "annotations": annotations}

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        image_id = self.image_ids[idx]
        image_info = self.images[image_id]
        anns = self.annotations_by_image.get(image_id, [])

        image_path = self.images_dir / image_info["file_name"]
        # 3 kanal (RGB) olarak yükle — termal görüntüleri 3 kanala genişletir.
        image = np.array(Image.open(image_path).convert("RGB"))

        boxes = [a["bbox"] for a in anns]
        categories = [a["category_id"] for a in anns]

        if self.transform is not None:
            transformed = self.transform(image=image, bboxes=boxes, category=categories)
            image = transformed["image"]
            boxes = transformed["bboxes"]
            categories = transformed["category"]

        formatted = self.format_image_annotations_as_coco(image_id, categories, boxes)
        result = self.image_processor(
            images=image, annotations=formatted, return_tensors="pt"
        )
        return {k: v[0] for k, v in result.items()}


Üç split için veri kümesi nesnelerini oluşturuyoruz.

In [ ]:
train_dataset = CocoDetectionDataset(TRAIN_DIR, train_coco, image_processor, transform=train_transform)
validation_dataset = CocoDetectionDataset(VAL_DIR, val_coco, image_processor, transform=validation_transform)
test_dataset = CocoDetectionDataset(TEST_DIR, test_coco, image_processor, transform=validation_transform)

print(f"Train: {len(train_dataset)} | Val: {len(validation_dataset)} | Test: {len(test_dataset)}")
sample = train_dataset[0]
print("pixel_values shape:", sample["pixel_values"].shape)
print("labels keys:", list(sample["labels"].keys()))

Ön işleme sonrası tensörlerin doğru şekilde oluşturulduğunu yukarıdaki çıktıyla doğrulayabilirsiniz. `pixel_values` `(3, image_size, image_size)` şeklindedir.

İsteğe bağlı: Ön işleme sonrası bir örneği görselleştirip bbox'ların hâlâ doğru olduğunu doğrulayalım.

In [ ]:
for i in [0, 1, 2]:
    sample = train_dataset[i]

    image = sample["pixel_values"]
    image = image.numpy().transpose(1, 2, 0)
    image = (image - image.min()) / (image.max() - image.min() + 1e-8) * 255.0
    image = Image.fromarray(image.astype(np.uint8))

    boxes = sample["labels"]["boxes"].numpy().copy()
    if boxes.size:
        # center_x, center_y, w, h (normalized) -> x, y, w, h (absolute)
        boxes[:, :2] = boxes[:, :2] - boxes[:, 2:] / 2
        w, h = image.size
        boxes = boxes * np.array([w, h, w, h])[None]

    categories_ = sample["labels"]["class_labels"].numpy()

    draw = ImageDraw.Draw(image)
    for box, category in zip(boxes, categories_):
        x, y, bw, bh = box
        draw.rectangle([x, y, x + bw, y + bh], outline="red", width=2)
        draw.text((x, y), id2label[int(category)], fill="white")

    display(image)

Son olarak, batch oluşturmak için özel bir `collate_fn` tanımlıyoruz.

In [ ]:
import torch

def collate_fn(batch):
    data = {}
    data["pixel_values"] = torch.stack([x["pixel_values"] for x in batch])
    data["labels"] = [x["labels"] for x in batch]
    return data

## Preparing function to compute mAP

Object detection models are commonly evaluated with a set of <a href="https://cocodataset.org/#detection-eval">COCO-style metrics</a>. We are going to use `torchmetrics` to compute `mAP` (mean average precision) and `mAR` (mean average recall) metrics and will wrap it to `compute_metrics` function in order to use in [Trainer](https://huggingface.co/docs/transformers/main/en/main_classes/trainer#transformers.Trainer) for evaluation.

Intermediate format of boxes used for training is `YOLO` (normalized) but we will compute metrics for boxes in `Pascal VOC` (absolute) format in order to correctly handle box areas. Let's define a function that converts bounding boxes to `Pascal VOC` format:

Then, in `compute_metrics` function we collect `predicted` and `target` bounding boxes, scores and labels from evaluation loop results and pass it to the scoring function.

In [ ]:
import numpy as np
from dataclasses import dataclass
from transformers.image_transforms import center_to_corners_format
from torchmetrics.detection.mean_ap import MeanAveragePrecision


@dataclass
class ModelOutput:
    logits: torch.Tensor
    pred_boxes: torch.Tensor


class MAPEvaluator:
    def __init__(self, image_processor, threshold=0.00, id2label=None):
        self.image_processor = image_processor
        self.threshold = threshold
        self.id2label = id2label

    def collect_image_sizes(self, targets):
        image_sizes = []
        for batch in targets:
            batch_image_sizes = torch.tensor(np.array([x["size"] for x in batch]))
            image_sizes.append(batch_image_sizes)
        return image_sizes

    def collect_targets(self, targets, image_sizes):
        post_processed_targets = []
        for target_batch, image_size_batch in zip(targets, image_sizes):
            for target, size in zip(target_batch, image_size_batch):
                height, width = size
                boxes = torch.tensor(target["boxes"])
                boxes = center_to_corners_format(boxes)
                boxes = boxes * torch.tensor([[width, height, width, height]])
                labels = torch.tensor(target["class_labels"])
                post_processed_targets.append({"boxes": boxes, "labels": labels})
        return post_processed_targets

    def collect_predictions(self, predictions, image_sizes):
        post_processed_predictions = []
        for batch, target_sizes in zip(predictions, image_sizes):
            batch_logits, batch_boxes = batch[1], batch[2]
            output = ModelOutput(logits=torch.tensor(batch_logits), pred_boxes=torch.tensor(batch_boxes))
            post_processed_output = self.image_processor.post_process_object_detection(
                output, threshold=self.threshold, target_sizes=target_sizes
            )
            post_processed_predictions.extend(post_processed_output)
        return post_processed_predictions

    @torch.no_grad()
    def __call__(self, evaluation_results):
        predictions, targets = evaluation_results.predictions, evaluation_results.label_ids
        image_sizes = self.collect_image_sizes(targets)
        post_processed_targets = self.collect_targets(targets, image_sizes)
        post_processed_predictions = self.collect_predictions(predictions, image_sizes)

        evaluator = MeanAveragePrecision(box_format="xyxy", class_metrics=True)
        evaluator.warn_on_many_detections = False
        evaluator.update(post_processed_predictions, post_processed_targets)
        metrics = evaluator.compute()

        classes = metrics.pop("classes")
        map_per_class = metrics.pop("map_per_class")
        mar_100_per_class = metrics.pop("mar_100_per_class")
        for class_id, class_map, class_mar in zip(classes, map_per_class, mar_100_per_class):
            class_name = self.id2label[class_id.item()] if self.id2label is not None else class_id.item()
            metrics[f"map_{class_name}"] = class_map
            metrics[f"mar_100_{class_name}"] = class_mar

        metrics = {k: round(v.item(), 4) for k, v in metrics.items()}
        return metrics


eval_compute_metrics_fn = MAPEvaluator(image_processor=image_processor, threshold=0.01, id2label=id2label)

## Training the detection model

You have done most of the heavy lifting in the previous sections, so now you are ready to train your model!
The images in this dataset are still quite large, even after resizing. This means that finetuning this model will
require at least one GPU.

Training involves the following steps:
1. Load the model with [AutoModelForObjectDetection](https://huggingface.co/docs/transformers/main/en/model_doc/auto#transformers.AutoModelForObjectDetection) using the same checkpoint as in the preprocessing.
2. Define your training hyperparameters in [TrainingArguments](https://huggingface.co/docs/transformers/main/en/main_classes/trainer#transformers.TrainingArguments).
3. Pass the training arguments to [Trainer](https://huggingface.co/docs/transformers/main/en/main_classes/trainer#transformers.Trainer) along with the model, dataset, image processor, and data collator.
4. Call [train()](https://huggingface.co/docs/transformers/main/en/main_classes/trainer#transformers.Trainer.train) to finetune your model.

When loading the model from the same checkpoint that you used for the preprocessing, remember to pass the `label2id`
and `id2label` maps that you created earlier from the dataset's metadata. Additionally, we specify `ignore_mismatched_sizes=True` to replace the existing classification head with a new one.

In [ ]:
from transformers import AutoModelForObjectDetection

model = AutoModelForObjectDetection.from_pretrained(
    checkpoint,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)
# Modelin girişi 3 kanal (RGB) olarak yapılandırılır (image_processor RGB normalizasyon yapar).
print("Num labels:", model.config.num_labels)

In the [TrainingArguments](https://huggingface.co/docs/transformers/main/en/main_classes/trainer#transformers.TrainingArguments) use `output_dir` to specify where to save your model, then configure hyperparameters as you see fit. For `num_train_epochs=10` training will take about 15 minutes in Google Colab T4 GPU, increase the number of epoch to get better results.

Important notes:
 - Do not remove unused columns because this will drop the image column. Without the image column, you
can't create `pixel_values`. For this reason, set `remove_unused_columns` to `False`.
 - Set `eval_do_concat_batches=False` to get proper evaluation results. Images have different number of target boxes, if batches are concatenated we will not be able to determine which boxes belongs to particular image.

If you wish to share your model by pushing to the Hub, set `push_to_hub` to `True` (you must be signed in to Hugging
Face to upload your model).

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="outputs/rtdetr_v2_r18vd_thermal",
    num_train_epochs=10,
    max_grad_norm=0.1,
    learning_rate=5e-5,
    warmup_steps=300,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    dataloader_num_workers=4,
    metric_for_best_model="eval_map",
    greater_is_better=True,
    load_best_model_at_end=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    remove_unused_columns=False,
    eval_do_concat_batches=False,
    fp16=True,           # GPU varsa hızlandırır; CPU'da False yapın.
    report_to="tensorboard",
    push_to_hub=False,
)

Finally, bring everything together, and call [train()](https://huggingface.co/docs/transformers/main/en/main_classes/trainer#transformers.Trainer.train):

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    processing_class=image_processor,
    data_collator=collate_fn,
    compute_metrics=eval_compute_metrics_fn,
)

trainer.train()

## Evaluate

In [ ]:
from pprint import pprint

metrics = trainer.evaluate(eval_dataset=test_dataset, metric_key_prefix="test")
pprint(metrics)

Modeli yerel `outputs/rtdetr_v2_r18vd_thermal` dizinine kaydedelim.

In [ ]:
save_dir = "outputs/rtdetr_v2_r18vd_thermal/final"
trainer.save_model(save_dir)
image_processor.save_pretrained(save_dir)
print("Saved to:", save_dir)

Hiperparametreleri (`num_train_epochs`, `learning_rate`, `per_device_train_batch_size`) görev bütçenize göre ayarlayabilirsiniz.

## Çıkarım (Inference)

Eğitilen modeli test kümesinden bir örnek üzerinde kullanalım.

In [ ]:
import torch
from PIL import Image, ImageDraw

device = "cuda" if torch.cuda.is_available() else "cpu"

# Test setinden örnek bir görüntü
sample_info = test_coco["images"][0]
image_path = TEST_DIR / "images" / sample_info["file_name"]
image = Image.open(image_path).convert("RGB")
print("Sample image:", image_path)

Eğitilmiş modeli yerel diskten yükleyin (veya yukarıda oturum içinde eğitilen modeli kullanın).

In [ ]:
from transformers import AutoImageProcessor, AutoModelForObjectDetection

# Oturum içindeki modeli kullanmak için bu hücreyi atlayın.
local_model_dir = "outputs/rtdetr_v2_r18vd_thermal/final"
image_processor = AutoImageProcessor.from_pretrained(local_model_dir)
model = AutoModelForObjectDetection.from_pretrained(local_model_dir)
model = model.to(device).eval()

Ve bounding box'ları tespit edelim:

In [ ]:
inputs = image_processor(images=[image], return_tensors="pt").to(device)
with torch.no_grad():
    outputs = model(**inputs)
target_sizes = torch.tensor([image.size[::-1]])

result = image_processor.post_process_object_detection(
    outputs, threshold=0.4, target_sizes=target_sizes
)[0]

for score, label, box in zip(result["scores"], result["labels"], result["boxes"]):
    box = [round(i, 2) for i in box.tolist()]
    print(
        f"Detected {model.config.id2label[label.item()]} with confidence "
        f"{round(score.item(), 3)} at location {box}"
    )

Sonucu görselleştirelim:

In [ ]:
image_with_boxes = image.copy()
draw = ImageDraw.Draw(image_with_boxes)

for score, label, box in zip(result["scores"], result["labels"], result["boxes"]):
    box = [round(i, 2) for i in box.tolist()]
    x, y, x2, y2 = tuple(box)
    draw.rectangle((x, y, x2, y2), outline="red", width=2)
    text_label = model.config.id2label[label.item()]
    draw.text((x, y), f"{text_label} [ {score.item():.2f} ]", fill="blue")

image_with_boxes